In [2]:
import serial
import serial.tools.list_ports
import time
from IPython.display import clear_output

def find_microcontroller_port():
    """Scans all available COM ports to find the microcontroller."""
    ports = serial.tools.list_ports.comports()
    keywords = ["Adafruit", "CircuitPython", "Arduino", "USB Serial", "CH340", "CP210x"]
    
    for port in ports:
        for keyword in keywords:
            if keyword.lower() in port.description.lower():
                return port.device
    if ports:
        return ports.device
    return None

com_port = find_microcontroller_port()

if not com_port:
    print("Error: No serial devices detected. Please check your USB cable connection.")
else:
    print(f"Target found on {com_port}. Initializing link...")    
    try:
        ser = serial.Serial(com_port, baudrate=115200, timeout=1)
        ser.reset_input_buffer() 
        time.sleep(0.1)
        print("Live data stream active. Press the Jupyter STOP button to halt.")
        
        # 3. Read loop
        while True:
            try:
                line = ser.readline().decode('utf-8').strip()
                if not line:
                    continue

                clean_line = line.replace('(', '').replace(')', '')
                parts = clean_line.split(',')
                
                if len(parts) == 3:
                    data_dict = {
                        'x': float(parts[0].strip()),
                        'y': float(parts[1].strip()),
                        'z': float(parts[2].strip())
                    }
                    
                    clear_output(wait=True)
                    print(data_dict)
                    
            except ValueError:
                continue
                
    except KeyboardInterrupt:
        print("\nStream safely paused by user.")
    except Exception as e:
        print(f"\nConnection error: {e}")
    finally:
        if 'ser' in locals() and ser.is_open:
            ser.close()
            print("Serial pipeline cleanly released.")


{'x': 3.86, 'y': -4.77, 'z': -7.33}

Connection error: ClearCommError failed (PermissionError(13, 'The device does not recognize the command.', None, 22))
Serial pipeline cleanly released.
